In [0]:
bronze_path = "abfss://bronze@marketingde2026.dfs.core.windows.net/amazon/amazon_ads.parquet"

df = spark.read.parquet(bronze_path)

display(df)

date,campaign_id,campaign_name,brand,product,impressions,clicks,spend,orders,sales
2026-08-01,AMZ001,Classmate_Search,Classmate,Notebook,173810,2956.0,46639.0,290,174196.0
2026-08-01,AMZ002,Classmate_Display,Classmate,Pens,119256,3071.0,93265.0,202,252964.0
2026-08-01,AMZ003,Verite_Launch,Classmate,Verite Notebook,101395,4918.0,72651.0,166,117811.0
2026-08-01,AMZ004,Stationery_Generic,Classmate,Stationery,102280,3395.0,60247.0,408,267815.0
2026-08-02,AMZ001,Classmate_Search,Classmate,Notebook,93478,4798.0,58031.0,429,219974.0
2026-08-02,AMZ002,Classmate_Display,Classmate,Pens,118893,4339.0,83618.0,292,111703.0
2026-08-02,AMZ003,Verite_Launch,Classmate,Verite Notebook,110926,5359.0,72696.0,324,182842.0
2026-08-02,AMZ004,Stationery_Generic,Classmate,Stationery,110379,null,107731.0,322,136793.0
2026-08-03,AMZ001,Classmate_Search,Classmate,Notebook,102156,4056.0,51338.0,333,200165.0
2026-08-03,AMZ002,Classmate_Display,Classmate,Pens,169131,3583.0,97895.0,172,230435.0


In [0]:
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- product: string (nullable = true)
 |-- impressions: string (nullable = true)
 |-- clicks: string (nullable = true)
 |-- spend: string (nullable = true)
 |-- orders: string (nullable = true)
 |-- sales: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, to_date

silver_df = (
    df
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    .withColumn("impressions", col("impressions").cast("long"))
    .withColumn("clicks", col("clicks").cast("double"))
    .withColumn("spend", col("spend").cast("double"))
    .withColumn("orders", col("orders").cast("long"))
    .withColumn("sales", col("sales").cast("double"))
)

silver_df.printSchema()

root
 |-- date: date (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- product: string (nullable = true)
 |-- impressions: long (nullable = true)
 |-- clicks: double (nullable = true)
 |-- spend: double (nullable = true)
 |-- orders: long (nullable = true)
 |-- sales: double (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum

null_counts = silver_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
])

display(null_counts)

date,campaign_id,campaign_name,brand,product,impressions,clicks,spend,orders,sales
0,0,0,0,0,0,1,1,0,1


In [0]:
display(
    null_counts.selectExpr(
        "stack(11, " +
        ", ".join([f"'{c}', {c}" for c in silver_df.columns]) +
        ") as (column_name, null_count)"
    )
)

column_name,null_count
date,0
campaign_id,0
campaign_name,0
brand,0
product,0
impressions,0
clicks,1
spend,1
orders,0
sales,1


In [0]:
display(
    silver_df.filter(
        col("clicks").isNull() |
        col("spend").isNull() |
        col("sales").isNull()
    )
)

date,campaign_id,campaign_name,brand,product,impressions,clicks,spend,orders,sales
2026-08-02,AMZ004,Stationery_Generic,Classmate,Stationery,110379,null,107731.0,322,136793.0
2026-08-05,AMZ003,Verite_Launch,Classmate,Verite Notebook,173886,5318.0,81500.0,262,null
2026-08-09,AMZ004,Stationery_Generic,Classmate,Stationery,116071,3126.0,null,232,251394.0


In [0]:
silver_df = silver_df.dropna(
    subset=["clicks", "spend", "sales"]
)

print("Rows after removing incomplete records:", silver_df.count())

print("Duplicate rows:", silver_df.count() - silver_df.dropDuplicates().count())

Rows after removing incomplete records: 122
Duplicate rows: 1


In [0]:
silver_df = silver_df.dropDuplicates()

print("Final Silver row count:", silver_df.count())

Final Silver row count: 121


In [0]:
silver_df = silver_df.dropDuplicates()

print("Rows after removing duplicates:", silver_df.count())

Rows after removing duplicates: 121


In [0]:
silver_path = "abfss://silver@marketingde2026.dfs.core.windows.net/amazon/amazon_ads.parquet"

silver_df.write \
    .mode("overwrite") \
    .parquet(silver_path)

print("Silver data written successfully")

Silver data written successfully


In [0]:
silver_check = spark.read.parquet(silver_path)

print("Silver row count:", silver_check.count())

display(silver_check)

Silver row count: 121


date,campaign_id,campaign_name,brand,product,impressions,clicks,spend,orders,sales
2026-08-15,AMZ004,Stationery_Generic,Classmate,Stationery,118016,4142.0,104142.0,180,153158.0
2026-08-29,AMZ002,Classmate_Display,Classmate,Pens,90053,3744.0,63803.0,257,222692.0
2026-08-05,AMZ004,Stationery_Generic,Classmate,Stationery,97331,3438.0,98857.0,166,192694.0
2026-08-12,AMZ003,Verite_Launch,Classmate,Verite Notebook,167128,3402.0,45471.0,186,275439.0
2026-08-24,AMZ003,Verite_Launch,Classmate,Verite Notebook,177806,5510.0,98133.0,277,179941.0
2026-08-04,AMZ004,Stationery_Generic,Classmate,Stationery,138520,3955.0,58730.0,286,279878.0
2026-08-06,AMZ004,Stationery_Generic,Classmate,Stationery,122325,5551.0,81789.0,425,178876.0
2026-08-08,AMZ003,Verite_Launch,Classmate,Verite Notebook,179353,4699.0,94209.0,286,278024.0
2026-08-10,AMZ002,Classmate_Display,Classmate,Pens,92552,2958.0,105889.0,335,190612.0
2026-08-13,AMZ003,Verite_Launch,Classmate,Verite Notebook,151953,3495.0,96420.0,392,216708.0
